In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

# Load feature engineered data
train = pd.read_csv('./data/processed/train_featured.csv')
test = pd.read_csv('./data/processed/test_featured.csv')

# Separate target variable
y_train = train['SalePrice']
train = train.drop(columns=['SalePrice'])

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('y_train shape:', y_train.shape)

Train shape: (1458, 88)
Test shape: (1459, 97)
y_train shape: (1458,)


In [2]:
# Fill any missing values before encoding
# Numerical columns - fill with median
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(test[col].median())

# Categorical columns - fill with mode
cat_cols = train.select_dtypes(include='object').columns
for col in cat_cols:
    train[col] = train[col].fillna(train[col].mode()[0])
    test[col] = test[col].fillna(test[col].mode()[0])

print('Missing values in train:', train.isnull().sum().sum())
print('Missing values in test:', test.isnull().sum().sum())

Missing values in train: 0
Missing values in test: 0


In [3]:
# Quality columns with natural order - Label Encoding
order_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
              'HeatingQC', 'KitchenQual', 'FireplaceQu',
              'GarageQual', 'GarageCond', 'PoolQC']

quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

for col in order_cols:
    if train[col].dtype == 'object':
        train[col] = train[col].map(quality_map)
        test[col] = test[col].map(quality_map)
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

print('Label encoding done!')
print(train[order_cols].isnull().sum())

Label encoding done!
ExterQual      0
ExterCond      0
BsmtQual       0
BsmtCond       0
HeatingQC      0
KitchenQual    0
FireplaceQu    0
GarageQual     0
GarageCond     0
PoolQC         0
dtype: int64


In [4]:
# One Hot Encoding for remaining categorical columns
remaining_cat = train.select_dtypes(include='object').columns
print('Remaining categorical columns to encode:', len(remaining_cat))

# Combine train and test before encoding
train_size = len(train)
combined = pd.concat([train, test], axis=0).reset_index(drop=True)

# Apply one hot encoding
combined = pd.get_dummies(combined, columns=remaining_cat, drop_first=True)

# Split back into train and test
train = combined[:train_size].reset_index(drop=True)
test = combined[train_size:].reset_index(drop=True)

print('Train shape after encoding:', train.shape)
print('Test shape after encoding:', test.shape)

Remaining categorical columns to encode: 43
Train shape after encoding: (1458, 261)
Test shape after encoding: (1459, 261)


In [5]:
# Fill any remaining NaN with 0 before scaling
train = train.fillna(0)
test = test.fillna(0)

print('Missing in train:', train.isnull().sum().sum())
print('Missing in test:', test.isnull().sum().sum())

Missing in train: 0
Missing in test: 0


In [6]:
# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train)
X_test_scaled = scaler.transform(test)

# Convert back to dataframes
X_train = pd.DataFrame(X_train_scaled, columns=train.columns)
X_test = pd.DataFrame(X_test_scaled, columns=test.columns)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (1458, 261)
X_test shape: (1459, 261)


In [7]:
# Save everything
os.makedirs('./data/processed', exist_ok=True)
os.makedirs('./models', exist_ok=True)

X_train.to_csv('./data/processed/X_train.csv', index=False)
X_test.to_csv('./data/processed/X_test.csv', index=False)
y_train.to_csv('./data/processed/y_train.csv', index=False)
joblib.dump(scaler, './models/scaler.pkl')

print('All files saved successfully!')
print('\nProcessed folder contents:')
print(os.listdir('./data/processed'))

All files saved successfully!

Processed folder contents:
['test_ids.csv', 'test_cleaned.csv', 'test_featured.csv', 'train_cleaned.csv', 'X_train.csv', 'y_train.csv', 'X_test.csv', 'train_featured.csv']
